In [1]:
import gazpacho

url = "https://en.wikipedia.org/wiki/List_of_world_records_in_swimming"
html = gazpacho.get(url)
soup = gazpacho.Soup.get(url)

In [2]:
# override find method to return list
type(soup.find(tag="none", mode="all"))

list

In [3]:
dir(soup)

['attrs', 'find', 'get', 'html', 'strip', 'tag', 'text']

In [4]:
all_tables = soup.find(tag="table", mode="all")

In [5]:
# total rows and columns for all tables identified
for num, table in enumerate(all_tables, 0):
  rows = table.find(tag="tr", mode="all")
  th_columns = rows[0].find(tag="th", mode="all")
  td_columns = rows[-1].find(tag="td", mode="all")
  print(f"{num} - {len(rows)} rows, {len(th_columns)} or {len(td_columns)} columns")

0 - 21 rows, 9 or 9 columns
1 - 21 rows, 9 or 9 columns
2 - 3 rows, 9 or 9 columns
3 - 25 rows, 9 or 9 columns
4 - 26 rows, 9 or 9 columns
5 - 4 rows, 9 or 9 columns
6 - 26 rows, 11 or 11 columns
7 - 1 rows, 8 or 2 columns
8 - 7 rows, 1 or 1 columns
9 - 5 rows, 1 or 1 columns
10 - 8 rows, 1 or 1 columns


- Data from Tables 0, 1 (50m pool) and 3, 4 (25m pool) are required for populating records under swimmer's charts.

In [6]:

table = all_tables[0]
for row in table.find(tag="tr", mode="all")[1:]:
  columns = row.find(tag="td", mode="all")
  event = columns[0].text
  time = columns[1].text
  print(event,'->',time)

50m freestyle -> 20.91
100m freestyle -> 46.40
200m freestyle -> 1:42.00
400m freestyle -> 3:39.96
800m freestyle -> 7:32.12
1500m freestyle -> 14:30.67
50m backstroke -> 23.55
100m backstroke -> 51.60
200m backstroke -> 1:51.92
50m breaststroke -> 25.95
100m breaststroke -> 56.88
200m breaststroke -> 2:05.48
50m butterfly -> 22.27
100m butterfly -> 49.45
200m butterfly -> 1:50.34
200m individual medley -> 1:52.69
400m individual medley -> 4:02.50
4 × 100 m freestyle relay -> 3:08.24
4 × 200 m freestyle relay -> 6:58.55
4 × 100 m medley relay -> 3:26.78


<hr> Tests Above <hr>
<hr> Prototype Below <hr>

In [7]:
COURSES = ("LC Men", "LC Women", "SC Men", "SC Women")
TABLES = (0, 1, 3, 4)

In [19]:
# prep data by building a dictionary-of-dictionaries
data = {}
for table, course in zip(TABLES, COURSES):
  rows = all_tables[table].find(tag="tr", mode="all")[1:] #skip <th> row
  data[course] = {}
  for row in rows:
    columns = row.find(tag="td", mode="all")
    event = columns[0].text
    time = columns[1].text
    if "relay" not in event:
      data[course][event] = time

In [9]:
data["LC Women"]

{'50m freestyle': '23.61',
 '100m freestyle': '51.71',
 '200m freestyle': '1:52.23',
 '400m freestyle': '3:54.18',
 '800m freestyle': '8:04.12',
 '1500m freestyle': '15:20.48',
 '50m backstroke': '26.86',
 '100m backstroke': '57.13',
 '200m backstroke': '2:03.14',
 '50m breaststroke': '29.16',
 '100m breaststroke': '1:04.13',
 '200m breaststroke': '2:17.55',
 '50m butterfly': '24.43',
 '100m butterfly': '54.60',
 '200m butterfly': '2:01.81',
 '200m individual medley': '2:05.70',
 '400m individual medley': '4:23.65'}

In [ ]:
# key mapping to account for differences between filenames and nested-dictionary keys in data
conversions = {
  "Free": "freestyle",
  "Back": "backstroke",
  "Fly":  "butterfly",
  "Breast": "breaststroke",
  "IM": "individual medley"
}

event = "Abi-10-50m-Back.txt"
*_, distance, stroke = event.removesuffix(".txt").split("-")
lookup = f"{distance} {conversions[stroke]}" # actual key format in courses
for course in COURSES:
  print(course, '->', data[course][lookup])

LC Men -> 23.55
LC Women -> 26.86
SC Men -> 22.11
SC Women -> 25.23


In [ ]:
# store data in a file-like object in JSON format
import json
with open("records.json", "w") as df:
  json.dump(obj=data, fp=df)

In [ ]:
# retrieve data from JSON file
with open("records.json", "r") as df:
  records = json.load(fp=df)

for course in COURSES:
  print(course, '->', records[course][lookup]) # distance, stroke from filename


LC Men -> 23.55
LC Women -> 26.86
SC Men -> 22.11
SC Women -> 25.23
